<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/10_GES_Aware_Genomic_RAG_Cell_7C3_Prompt_Materialization_Execution_Authorization_V3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Google Colab; Drive mount skipped.')

ROOT = Path('/content/drive/MyDrive/GES_RAG_Temporal_Study')
if not ROOT.exists():
    raise FileNotFoundError(
        f'Project root not found: {ROOT}\n'
        'Confirm that Google Drive is mounted and the project folder is unchanged.'
    )

print(f'Project root: {ROOT}')

Mounted at /content/drive
Project root: /content/drive/MyDrive/GES_RAG_Temporal_Study


## 1. Imports, frozen identities, and exact expected checksums

In [2]:
from __future__ import annotations

from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import csv
import hashlib
import json
import re

import pyarrow.parquet as pq

NOTEBOOK_NAME = '10_GES_Aware_Genomic_RAG_Cell_7C3_Prompt_Materialization_Execution_Authorization_V3.ipynb'
CELL_ID = '7C3'
STAGE = '7C'
PACKAGE_VERSION = 'v1'
CREATED_UTC = datetime.now(timezone.utc).isoformat()

EXPECTED_QUESTIONS = 80
EXPECTED_ALIASES = 6
EXPECTED_TOP5_PER_PROMPT = 5
EXPECTED_PROMPTS = 480
EXPECTED_SCORE_BLIND_TOP5_ROWS = 2400

EXPECTED_CELL_7C2_TERMINAL_DECISION = (
    'PASS_STAGE7C2_FROZEN_CELL7A3_SCORES_JOINED_TO_EXACT_CELL7C0_TOP20_'
    'SIX_CONDITION_QUALITY_RANKS_FIXED_075_SEMANTIC_025_QUALITY_RRF_CONSTANT60_'
    'AND_2400_SCORE_BLIND_FINAL_TOP5_SELECTIONS_MATERIALIZED_CHECKSUM_PROTECTED_'
    'NO_NEW_RETRIEVAL_HARD_EXCLUSION_PROMPTS_LLM_ANSWER_KEYS_ADJUDICATION_OR_RAG_METRICS_'
    'NEXT_EXECUTION_NOT_AUTHORIZED'
)

EXPECTED_CELL_7B4_DECISION = (
    'PASS_STAGE7B4_EXACT_EMBEDDING_MODEL_REVISION_TEXT_NORMALIZATION_'
    'SIMILARITY_TOP20_CANDIDATE_POOL_TOP5_CONTEXT_QUALITY_RERANKING_'
    'BLINDED_ALIASES_PROMPTS_STRICT_RESPONSE_SCHEMA_FIXED_LLM_SNAPSHOT_'
    'GENERATION_RUNTIME_AND_DETERMINISTIC_CONTROLS_FROZEN_CHECKSUM_PROTECTED_'
    'NO_EMBEDDINGS_RETRIEVAL_RERANKING_PROMPT_MATERIALIZATION_LLM_'
    'ANSWER_KEY_OUTCOME_INSPECTION_OR_RAG_EVALUATION_EXECUTION_NOT_AUTHORIZED'
)

# --------------------------------------------------------------------------------------------------
# Exact successful Cell 7C2 package from the terminal PASS.
# --------------------------------------------------------------------------------------------------
CELL_7C2_EXEC_DIR = (
    ROOT / 'outputs' / 'rag_execution' / 'stage7_rag'
    / 'cell_7c2_quality_reranking_and_top5_v1'
)
CELL_7C2_QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'cell_7c2_quality_reranking_and_top5_v1'
)
CELL_7C2_CONFIG_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c2_quality_reranking_and_top5_v1'
)

CELL_7C2 = OrderedDict([
    ('ranking_audit', {
        'path': CELL_7C2_EXEC_DIR / 'cell_7c2_six_condition_quality_reranking_audit_v1.parquet',
        'sha256': '394b96dc8c05a996a180ddb949d5661ca40f51220a24f94bd6702795c23ebf5d',
    }),
    ('score_blind_top5', {
        'path': CELL_7C2_EXEC_DIR / 'cell_7c2_score_blind_final_top5_context_selection_v1.parquet',
        'sha256': '88abb743262fe1dd2fede29f613893fd6a8a8a67c7d56efa75a961bf6d259c72',
    }),
    ('condition_summary', {
        'path': CELL_7C2_EXEC_DIR / 'cell_7c2_condition_reranking_summary_v1.csv',
        'sha256': '09ebf75a62f955904b12c56dc338956aaf9b63b5ea0337d40dea3da4b65aeea9',
    }),
    ('input_inventory', {
        'path': CELL_7C2_CONFIG_DIR / 'cell_7c2_verified_input_inventory_v1.csv',
        'sha256': '28eec1700bb43aa252007e0f5fb2d70487d16b0d25f8b2ea010765b46b9cc199',
    }),
    ('execution_report', {
        'path': CELL_7C2_QC_DIR / 'cell_7c2_quality_reranking_execution_report_v1.json',
        'sha256': 'a6c5a1053cad4b144f280dc0c65152407a6954ce9bc7684867fea0566ad685fb',
    }),
    ('qc', {
        'path': CELL_7C2_QC_DIR / 'cell_7c2_quality_reranking_qc_v1.json',
        'sha256': '096fdf19e044eafceffe8c1affb1590d8fdb60b8f3444f57032e9b198462dbf0',
    }),
    ('manifest', {
        'path': CELL_7C2_CONFIG_DIR / 'cell_7c2_quality_reranking_and_top5_manifest_v1.json',
        'sha256': '8a8e3dc827b4641f28922b8712c3024fb4c82ba1e3b4511213f28b9b4b1a83d5',
    }),
])

# --------------------------------------------------------------------------------------------------
# Cell 7B3 score-blind prompt inputs. Resolve by exact filename + exact checksum so directory
# assumptions cannot silently select the wrong copy.
# --------------------------------------------------------------------------------------------------
CELL_7B3_SCORE_BLIND = OrderedDict([
    ('semantic_corpus', {
        'filename': 'cell_7b3_score_blind_semantic_corpus_v1.parquet',
        'sha256': '2fead04f6c0814bb87207c9c36db7475370ae626f6a326c89ce672f402339399',
        'expected_rows': 100_920,
    }),
    ('primary_questions', {
        'filename': 'cell_7b3_primary_question_set_v1.csv',
        'sha256': 'c76e81952fcc6a698866b64da7b7daabeb281b7b9d10e17873596095b69d95df',
        'expected_rows': 80,
    }),
])

# --------------------------------------------------------------------------------------------------
# Cell 7B4 exact prompt/LLM/runtime freeze.
# --------------------------------------------------------------------------------------------------
CELL_7B4_CONFIG_DIR = ROOT / 'configs' / 'stage7_rag' / 'cell_7b4_configuration_freeze_v1'
CELL_7B4_QC_DIR = ROOT / 'outputs' / 'quality_checks' / 'stage7_rag' / 'cell_7b4_configuration_freeze_v1'

CELL_7B4 = OrderedDict([
    ('llm_prompt_response', {
        'path': CELL_7B4_CONFIG_DIR / 'cell_7b4_llm_prompt_response_configuration_v1.json',
        'sha256': 'e3f684f9c471b8074dc41f2398f8aff0cb03217f188e2eab2ad8f4c0070a810b',
    }),
    ('runtime_determinism', {
        'path': CELL_7B4_CONFIG_DIR / 'cell_7b4_runtime_and_determinism_configuration_v1.json',
        'sha256': '6003c85ef151ae1d7dca530462fa1dbcf6983be4b7e92744b8e65c8d8b42b1d3',
    }),
    ('condition_aliases', {
        'path': CELL_7B4_CONFIG_DIR / 'cell_7b4_condition_alias_inventory_v1.csv',
        'sha256': '6eb45683b42a456d2b6788a5fcf6b9cd95fc606afe9627610ebbc11914312cb9',
    }),
    ('requirements_lock', {
        'path': CELL_7B4_CONFIG_DIR / 'cell_7b4_execution_requirements_lock_v1.txt',
        'sha256': '1a894f7ba976d00563325cc3324ff91708b5699e1c618e3674502fe586828a91',
    }),
    ('qc', {
        'path': CELL_7B4_QC_DIR / 'cell_7b4_configuration_freeze_qc_v1.json',
        'sha256': '01d6ef836c25d49b9df32d2739d1d532a1f205be0d83c6227eb65070d9f9ced4',
    }),
    ('manifest', {
        'path': CELL_7B4_CONFIG_DIR / 'cell_7b4_configuration_freeze_manifest_v1.json',
        'sha256': '18a5d6cb3cadca0eab950839a19022686fc6bad2c398ed87f2a48c66eff462fe',
    }),
])

AUTH_DIR = ROOT / 'configs' / 'stage7_rag' / 'cell_7c3_prompt_materialization_authorization_v2'
QC_DIR = ROOT / 'outputs' / 'quality_checks' / 'stage7_rag' / 'cell_7c3_prompt_materialization_authorization_v2'

OUTPUTS = OrderedDict([
    ('authorization',
     AUTH_DIR / 'cell_7c3_stage7c_cell7c4_prompt_materialization_authorization_v2.json'),
    ('input_inventory',
     AUTH_DIR / 'cell_7c3_authorized_prompt_materialization_input_inventory_v2.csv'),
    ('qc',
     QC_DIR / 'cell_7c3_prompt_materialization_authorization_qc_v2.json'),
    ('manifest',
     AUTH_DIR / 'cell_7c3_prompt_materialization_authorization_manifest_v2.json'),
])

for directory in (AUTH_DIR, QC_DIR):
    directory.mkdir(parents=True, exist_ok=True)

if OUTPUTS['manifest'].exists():
    raise FileExistsError(
        f'Cell 7C3 manifest already exists: {OUTPUTS["manifest"]}\\n'
        'Fail-closed overwrite protection is active.'
    )

print(f'Authorization directory: {AUTH_DIR}')
print(f'QC directory           : {QC_DIR}')

Authorization directory: /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage7_rag/cell_7c3_prompt_materialization_authorization_v2
QC directory           : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/quality_checks/stage7_rag/cell_7c3_prompt_materialization_authorization_v2


## 2. Strict checksum, sidecar, metadata, and serialization helpers

In [3]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def sidecar_path(path: Path) -> Path:
    return path.with_name(path.name + '.sha256')


def read_sidecar_hash(path: Path) -> str:
    """Read the first SHA-256 token from a standard '<hash>  <filename>' sidecar."""
    text = path.read_text(encoding='utf-8').strip()
    if not text:
        raise ValueError(f'Empty SHA-256 sidecar: {path}')

    token = text.split()[0].strip()
    if not re.fullmatch(r'[0-9a-fA-F]{64}', token):
        raise ValueError(
            f'Invalid SHA-256 sidecar format: {path}\n'
            f'Observed first token: {token!r}'
        )
    return token.lower()


def sidecar_is_valid(path: Path) -> bool:
    sc = sidecar_path(path)
    return (
        path.exists()
        and sc.exists()
        and read_sidecar_hash(sc) == sha256_file(path)
    )


def verify_exact_artifact(label: str, path: Path, expected_sha256: str) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f'Missing {label}: {path}')
    observed = sha256_file(path)
    if observed != expected_sha256:
        raise AssertionError(
            f'SHA-256 mismatch for {label}.\\n'
            f'Expected: {expected_sha256}\\nObserved: {observed}\\nPath: {path}'
        )
    if not sidecar_is_valid(path):
        raise AssertionError(f'Invalid or missing SHA-256 sidecar for {label}: {path}')
    return {
        'input_id': label,
        'path': str(path),
        'sha256': observed,
        'bytes': int(path.stat().st_size),
        'sidecar_path': str(sidecar_path(path)),
        'sidecar_valid': True,
    }


def locate_exact_hash(filename: str, expected_sha256: str) -> Path:
    candidates = sorted(path for path in ROOT.rglob(filename) if path.is_file())
    matches = [path for path in candidates if sha256_file(path) == expected_sha256]
    if len(matches) != 1:
        details = '\\n'.join(str(path) for path in candidates) or '<none>'
        raise RuntimeError(
            f'Expected exactly one checksum-matching artifact for {filename}; '
            f'found {len(matches)}.\\nCandidates:\\n{details}'
        )
    return matches[0]


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding='utf-8'))


def parquet_metadata(path: Path) -> dict[str, Any]:
    pf = pq.ParquetFile(path)
    return {
        'rows': int(pf.metadata.num_rows),
        'columns': int(pf.metadata.num_columns),
        'schema_names': list(pf.schema_arrow.names),
    }


def csv_header_and_row_count(path: Path) -> tuple[list[str], int]:
    with path.open('r', encoding='utf-8', newline='') as handle:
        reader = csv.reader(handle)
        try:
            header = next(reader)
        except StopIteration:
            raise AssertionError(f'Empty CSV: {path}')
        rows = sum(1 for _ in reader)
    return header, rows


def to_json_native(value: Any) -> Any:
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(k): to_json_native(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [to_json_native(v) for v in value]
    if hasattr(value, 'item'):
        return to_json_native(value.item())
    raise TypeError(f'Unsupported JSON type: {type(value).__name__}')


def stable_write_json(path: Path, payload: Any) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    native = to_json_native(payload)
    text = json.dumps(
        native,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
        allow_nan=False,
    ) + '\n'
    path.write_text(text, encoding='utf-8')
    return sha256_file(path)


def stable_write_csv(path: Path, rows: list[dict[str, Any]], fieldnames: list[str]) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames, lineterminator='\n')
        writer.writeheader()
        for row in rows:
            writer.writerow({key: to_json_native(row.get(key, '')) for key in fieldnames})
    return sha256_file(path)


def write_sidecar(path: Path) -> None:
    digest = sha256_file(path)
    sidecar_path(path).write_text(f'{digest}  {path.name}\n', encoding='utf-8')


# Parser self-test: validates the exact sidecar text format written by Stage 7 cells.
import tempfile as _tempfile

with _tempfile.TemporaryDirectory(prefix='cell_7c3_sidecar_parser_test_') as _tmp:
    _test_path = Path(_tmp) / 'artifact.bin'
    _test_path.write_bytes(b'cell-7c3-sidecar-parser-self-test')
    _test_hash = sha256_file(_test_path)
    _test_sidecar = sidecar_path(_test_path)
    _test_sidecar.write_text(
        f'{_test_hash}  {_test_path.name}\n',
        encoding='utf-8',
    )
    if read_sidecar_hash(_test_sidecar) != _test_hash:
        raise AssertionError('SHA-256 sidecar parser self-test failed.')
    if not sidecar_is_valid(_test_path):
        raise AssertionError('SHA-256 sidecar validation self-test failed.')

print('SHA-256 sidecar parser self-test: PASS')

# Serialization self-test: fail before touching frozen Cell 7C3 outputs.
import tempfile as _writer_tempfile

with _writer_tempfile.TemporaryDirectory(prefix='cell_7c3_writer_test_') as _tmp:
    _tmpdir = Path(_tmp)

    _json_path = _tmpdir / 'test.json'
    _json_payload = {'ok': True, 'nested': {'value': 7}}
    stable_write_json(_json_path, _json_payload)
    _json_roundtrip = json.loads(_json_path.read_text(encoding='utf-8'))
    if _json_roundtrip != _json_payload:
        raise AssertionError('JSON serialization round-trip self-test failed.')
    _json_bytes = _json_path.read_bytes()
    if not _json_bytes.endswith(b'\n'):
        raise AssertionError('JSON writer did not terminate with a real newline byte.')
    if _json_bytes.endswith(b'\\n'):
        raise AssertionError('JSON writer emitted a literal backslash+n suffix.')

    _csv_path = _tmpdir / 'test.csv'
    stable_write_csv(
        _csv_path,
        [{'a': 1, 'b': 2}, {'a': 3, 'b': 4}],
        ['a', 'b'],
    )
    _csv_text = _csv_path.read_text(encoding='utf-8')
    if '\n' not in _csv_text or '\\n' in _csv_text:
        raise AssertionError('CSV writer newline self-test failed.')

    write_sidecar(_json_path)
    if not sidecar_is_valid(_json_path):
        raise AssertionError('Sidecar writer/reader self-test failed.')

print('JSON/CSV/sidecar serialization self-test: PASS')

print('Strict authorization helpers loaded.')

SHA-256 sidecar parser self-test: PASS
JSON/CSV/sidecar serialization self-test: PASS
Strict authorization helpers loaded.


## 3. Reverify the complete successful Cell 7C2 package without opening the score-bearing audit

In [4]:
verified_7c2 = OrderedDict()

for artifact_id, spec in CELL_7C2.items():
    verified_7c2[artifact_id] = verify_exact_artifact(
        f'cell_7c2_{artifact_id}',
        spec['path'],
        spec['sha256'],
    )

manifest_7c2 = load_json(CELL_7C2['manifest']['path'])
qc_7c2 = load_json(CELL_7C2['qc']['path'])

if manifest_7c2.get('terminal_decision') != EXPECTED_CELL_7C2_TERMINAL_DECISION:
    raise AssertionError(
        'Cell 7C2 terminal decision mismatch.\\n'
        f'Observed: {manifest_7c2.get("terminal_decision")}'
    )
if manifest_7c2.get('next_authorized_cell') is not None:
    raise AssertionError('Cell 7C2 unexpectedly authorizes a downstream execution cell.')
if int(qc_7c2.get('failed_checks', -1)) != 0:
    raise AssertionError('Cell 7C2 QC does not report zero failures.')

top5_meta = parquet_metadata(CELL_7C2['score_blind_top5']['path'])
expected_top5_columns = [
    'question_id',
    'blinded_alias',
    'context_position',
    'corpus_row_index',
    'packet_id',
    'rcv_accession',
]
if top5_meta['rows'] != EXPECTED_SCORE_BLIND_TOP5_ROWS:
    raise AssertionError(
        f'Cell 7C2 score-blind top-5 rows changed: {top5_meta["rows"]}'
    )
if top5_meta['schema_names'] != expected_top5_columns:
    raise AssertionError(
        'Cell 7C2 score-blind top-5 schema changed.\\n'
        f'Observed: {top5_meta["schema_names"]}'
    )

prohibited_top5_tokens = [
    'ges', 'quality', 'semantic_rank', 'semantic_score', 'rrf',
    'condition_id', 'condition_name', 'condition_role',
    'review', 'conflict', 'stable', 'instability',
]
leaking_top5_columns = [
    column
    for column in top5_meta['schema_names']
    if any(token in column.lower() for token in prohibited_top5_tokens)
]
if leaking_top5_columns:
    raise AssertionError(
        'Cell 7C2 score-blind top-5 contains prohibited columns: '
        + ', '.join(leaking_top5_columns)
    )

# Deliberately do NOT open the ranking_audit Parquet row content.
ranking_audit_meta = parquet_metadata(CELL_7C2['ranking_audit']['path'])
if ranking_audit_meta['rows'] != 9_600:
    raise AssertionError('Cell 7C2 audit metadata row count changed.')

print('Cell 7C2 complete package                : 7/7 exact hashes + sidecars')
print('Cell 7C2 terminal PASS                  : VERIFIED')
print(f'Score-blind final top-5 metadata        : {top5_meta["rows"]:,} rows × {top5_meta["columns"]} columns')
print('Score-bearing Cell 7C2 audit rows opened: NO')
print('Answer-key outcomes inspected           : NO')

Cell 7C2 complete package                : 7/7 exact hashes + sidecars
Cell 7C2 terminal PASS                  : VERIFIED
Score-blind final top-5 metadata        : 2,400 rows × 6 columns
Score-bearing Cell 7C2 audit rows opened: NO
Answer-key outcomes inspected           : NO


## 4. Reverify the exact Cell 7B3 score-blind prompt sources and Cell 7B4 prompt/LLM freeze

In [5]:
verified_inputs: list[dict[str, Any]] = []

# Only the two score-blind Cell 7B3 data sources required for prompt construction.
resolved_7b3 = {}
for key, spec in CELL_7B3_SCORE_BLIND.items():
    path = locate_exact_hash(spec['filename'], spec['sha256'])
    resolved_7b3[key] = path
    record = verify_exact_artifact(
        f'cell_7b3_{key}',
        path,
        spec['sha256'],
    )
    record['source_cell'] = '7B3'
    verified_inputs.append(record)

semantic_meta = parquet_metadata(resolved_7b3['semantic_corpus'])
if semantic_meta['rows'] != 100_920:
    raise AssertionError(f'Semantic corpus rows changed: {semantic_meta["rows"]}')
required_semantic_columns = {
    'evidence_packet_id',
    'rcv_accession',
    'semantic_evidence_text',
}
if not required_semantic_columns.issubset(set(semantic_meta['schema_names'])):
    raise AssertionError(
        'Semantic corpus is missing required score-blind prompt fields.\\n'
        f'Observed schema: {semantic_meta["schema_names"]}'
    )

question_header, question_rows = csv_header_and_row_count(resolved_7b3['primary_questions'])
if question_rows != 80:
    raise AssertionError(f'Primary-question row count changed: {question_rows}')
if not {'question_id', 'question_text'}.issubset(set(question_header)):
    raise AssertionError(
        'Primary-question CSV is missing question_id/question_text.\\n'
        f'Header: {question_header}'
    )

# Exact frozen prompt/LLM configuration package.
verified_7b4 = OrderedDict()
for artifact_id, spec in CELL_7B4.items():
    record = verify_exact_artifact(
        f'cell_7b4_{artifact_id}',
        spec['path'],
        spec['sha256'],
    )
    record['source_cell'] = '7B4'
    verified_7b4[artifact_id] = record
    verified_inputs.append(record)

llm_config = load_json(CELL_7B4['llm_prompt_response']['path'])
runtime_config = load_json(CELL_7B4['runtime_determinism']['path'])
manifest_7b4 = load_json(CELL_7B4['manifest']['path'])

if manifest_7b4.get('terminal_decision') != EXPECTED_CELL_7B4_DECISION:
    raise AssertionError('Cell 7B4 terminal decision mismatch.')

llm = llm_config.get('llm', {})
generation = llm_config.get('generation', {})
prompt_cfg = llm_config.get('prompt', {})
structured = llm_config.get('structured_output', {})
schema = structured.get('schema', {})

expected_required_fields = {
    'answer',
    'clinical_significance',
    'conflict_detected',
    'evidence_strength',
    'response_policy',
    'confidence',
    'evidence_ids',
    'reasoning_summary',
}

frozen_prompt_checks = OrderedDict([
    ('llm_provider_openai', llm.get('provider') == 'OpenAI'),
    ('responses_api_exact', llm.get('api') == 'Responses API'),
    ('fixed_model_snapshot_exact', llm.get('model') == 'gpt-4.1-mini-2025-04-14'),
    ('model_aliases_prohibited', set(llm.get('model_aliases_prohibited', [])) == {'gpt-4.1-mini', 'latest'}),
    ('fixed_snapshot_required', llm.get('fixed_snapshot_required') is True),
    ('tools_disabled', llm.get('tools') == []),
    ('web_search_disabled', llm.get('web_search') is False),
    ('file_search_disabled', llm.get('file_search') is False),
    ('code_interpreter_disabled', llm.get('code_interpreter') is False),
    ('store_false', llm.get('store') is False),
    ('stream_false', llm.get('stream') is False),
    ('temperature_zero', generation.get('temperature') == 0.0),
    ('top_p_one', generation.get('top_p') == 1.0),
    ('max_output_tokens_1200', generation.get('max_output_tokens') == 1200),
    ('presence_penalty_zero', generation.get('presence_penalty') == 0.0),
    ('frequency_penalty_zero', generation.get('frequency_penalty') == 0.0),
    ('three_repetitions', generation.get('repetitions_per_question_condition') == 3),
    ('run_ids_0_1_2', generation.get('run_ids') == [0, 1, 2]),
    ('question_before_context', prompt_cfg.get('question_position') == 'before evidence context'),
    ('context_order_frozen_top5', prompt_cfg.get('context_order') == 'frozen final top-5 order'),
    ('score_values_not_in_prompt', prompt_cfg.get('score_values_in_prompt') is False),
    ('condition_identity_not_in_prompt', prompt_cfg.get('condition_identity_in_prompt') is False),
    ('answer_key_not_in_prompt', prompt_cfg.get('answer_key_in_prompt') is False),
    ('question_placeholder_present', '{question_text}' in str(prompt_cfg.get('user_prompt_template', ''))),
    ('context_placeholder_present', '{context_blocks}' in str(prompt_cfg.get('user_prompt_template', ''))),
    ('strict_json_schema', structured.get('strict') is True),
    ('json_schema_type_exact', structured.get('type') == 'json_schema'),
    ('response_schema_no_additional_properties', schema.get('additionalProperties') is False),
    ('response_schema_all_eight_fields_required', set(schema.get('required', [])) == expected_required_fields),
    ('response_schema_properties_exact', set(schema.get('properties', {}).keys()) == expected_required_fields),
])

failed_prompt_checks = [
    name for name, passed in frozen_prompt_checks.items()
    if not bool(passed)
]
if failed_prompt_checks:
    raise RuntimeError(
        'Frozen Cell 7B4 prompt/LLM configuration verification failed:\\n- '
        + '\\n- '.join(failed_prompt_checks)
    )

for required_text_key in (
    'system_prompt',
    'user_prompt_template',
    'context_block_template',
):
    if not isinstance(prompt_cfg.get(required_text_key), str) or not prompt_cfg[required_text_key].strip():
        raise AssertionError(f'Missing frozen prompt text: {required_text_key}')

for text_key, hash_key in (
    ('system_prompt', 'system_prompt_sha256'),
    ('user_prompt_template', 'user_prompt_template_sha256'),
    ('context_block_template', 'context_block_template_sha256'),
):
    observed = hashlib.sha256(prompt_cfg[text_key].encode('utf-8')).hexdigest()
    if observed != prompt_cfg.get(hash_key):
        raise AssertionError(f'Frozen prompt-text hash mismatch: {text_key}')

for forbidden in ('ges_score', 'p_stable', 'quality_rank', 'semantic_rank', 'rrf'):
    if forbidden in prompt_cfg['user_prompt_template'].lower():
        raise AssertionError(
            f'Frozen user prompt unexpectedly contains a prohibited score/rank token: {forbidden}'
        )

print(f'Score-blind semantic corpus             : {semantic_meta["rows"]:,} rows; exact checksum verified')
print(f'Primary questions                       : {question_rows} rows; exact checksum verified')
print('Cell 7B4 prompt/LLM configuration       : VERIFIED')
print('Frozen model snapshot                   : gpt-4.1-mini-2025-04-14')
print('Frozen generations per question/alias   : 3')
print('Prompt score exposure                   : NO')
print('Prompt condition identity exposure      : NO')
print('Answer-key outcomes inspected           : NO')

Score-blind semantic corpus             : 100,920 rows; exact checksum verified
Primary questions                       : 80 rows; exact checksum verified
Cell 7B4 prompt/LLM configuration       : VERIFIED
Frozen model snapshot                   : gpt-4.1-mini-2025-04-14
Frozen generations per question/alias   : 3
Prompt score exposure                   : NO
Prompt condition identity exposure      : NO
Answer-key outcomes inspected           : NO


## 5. Freeze narrow authorization for Cell 7C4 prompt materialization only

In [6]:
authorization_decision = (
    'AUTHORIZE_STAGE7C_CELL7C4_SCORE_BLIND_PROMPT_MATERIALIZATION_ONLY_'
    '480_QUESTION_ALIAS_PROMPTS_FIVE_FROZEN_CONTEXT_BLOCKS_EACH_EXACT_CELL7B4_TEMPLATES_'
    'NO_SCORE_BEARING_AUDIT_GES_QUALITY_RANK_RRF_CONDITION_IDENTITY_ANSWER_KEYS_LLM_OR_METRICS'
)

AUTHORIZED_CELL = {
    'stage': '7C',
    'cell_id': '7C4',
    'title': 'Score-blind prompt materialization from frozen final top-5 contexts',
    'authorized_once': True,
    'overwrite_existing_outputs': False,
}

AUTHORIZED_OPERATIONS = [
    'Load the exact checksum-frozen Cell 7C2 score-blind final top-5 selection artifact.',
    'Load the exact checksum-frozen Cell 7B3 score-blind semantic corpus.',
    'Load the exact checksum-frozen Cell 7B3 primary-question set.',
    'Load the exact checksum-frozen Cell 7B4 prompt/response configuration and blinded aliases.',
    'Join every selected packet_id to exactly one semantic-corpus row and preserve the exact RCV identity.',
    'Join every question_id to exactly one frozen primary question.',
    'Materialize exactly 480 question-by-blinded-alias prompt instances.',
    'Materialize exactly five context blocks per prompt in context_position 1 through 5 order.',
    'Use the exact Cell 7B4 system prompt, user prompt template, and context-block template without modification.',
    'Preserve blinded_alias only; do not materialize A-F condition IDs or condition names in the prompt package.',
    'Compute and freeze deterministic system-prompt, user-prompt, full-prompt, and context hashes.',
    'Write checksum sidecars, prompt-materialization QC, execution report, and manifest.',
]

PROHIBITED_OPERATIONS = [
    'Load row-level content from the Cell 7C2 score-bearing quality-reranking audit.',
    'Load Cell 7A3 score rows.',
    'Expose Full-GES, no-star-GES, combined-metadata, quality-signal, quality-rank, semantic-rank, semantic-score, RRF, or random-quality values to prompts.',
    'Expose A-F condition IDs, condition names, or condition roles to prompts.',
    'Change final top-5 membership or context order.',
    'Change any frozen Cell 7B4 prompt text, response schema, model snapshot, generation setting, or blinded alias.',
    'Load or inspect structured answer-key outcomes.',
    'Call the OpenAI Responses API or any other LLM.',
    'Generate or inspect model responses.',
    'Perform adjudication.',
    'Calculate retrieval, answer, RAG, bootstrap, or statistical performance metrics.',
    'Modify prompts after reviewing any test-set LLM response.',
]

input_records = []

# Cell 7C2 inputs that are cryptographically required by the authorization lineage.
for artifact_id in ('score_blind_top5', 'manifest', 'qc', 'execution_report'):
    record = dict(verified_7c2[artifact_id])
    record['source_cell'] = '7C2'
    record['row_level_content_opened_in_cell_7c3'] = False
    input_records.append(record)

for key, path in resolved_7b3.items():
    spec = CELL_7B3_SCORE_BLIND[key]
    record = {
        'input_id': f'cell_7b3_{key}',
        'source_cell': '7B3',
        'path': str(path),
        'sha256': spec['sha256'],
        'bytes': int(path.stat().st_size),
        'sidecar_path': str(sidecar_path(path)),
        'sidecar_valid': True,
        'row_level_content_opened_in_cell_7c3': False,
    }
    input_records.append(record)

for artifact_id in ('llm_prompt_response', 'runtime_determinism', 'condition_aliases', 'requirements_lock', 'manifest'):
    record = dict(verified_7b4[artifact_id])
    record['source_cell'] = '7B4'
    record['row_level_content_opened_in_cell_7c3'] = False
    input_records.append(record)

authorization_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'notebook': NOTEBOOK_NAME,
    'project_root': str(ROOT),
    'authorization_type': 'fail_closed_scientific_execution_authorization',
    'authorization_decision': authorization_decision,
    'authorized_cell': AUTHORIZED_CELL,
    'authorized_operations': AUTHORIZED_OPERATIONS,
    'prohibited_operations': PROHIBITED_OPERATIONS,
    'frozen_prompt_materialization_design': {
        'questions': EXPECTED_QUESTIONS,
        'blinded_aliases': EXPECTED_ALIASES,
        'prompt_instances': EXPECTED_PROMPTS,
        'context_blocks_per_prompt': EXPECTED_TOP5_PER_PROMPT,
        'selected_context_rows': EXPECTED_SCORE_BLIND_TOP5_ROWS,
        'question_position': 'before evidence context',
        'context_order': 'frozen final top-5 order',
        'score_values_in_prompt': False,
        'condition_identity_in_prompt': False,
        'answer_key_in_prompt': False,
        'llm_call_authorized': False,
        'model_snapshot_reserved_for_later_generation': 'gpt-4.1-mini-2025-04-14',
        'future_generation_repetitions_per_question_alias': 3,
        'future_generation_run_ids': [0, 1, 2],
        'future_expected_response_calls_if_separately_authorized': 1440,
    },
    'verified_inputs': input_records,
    'cell_7c2_score_bearing_audit_rows_opened': False,
    'cell_7a3_scores_loaded': False,
    'answer_key_outcomes_inspected': False,
    'prompts_materialized': False,
    'llm_called': False,
    'rag_metrics_calculated': False,
}

stable_write_json(OUTPUTS['authorization'], authorization_payload)
write_sidecar(OUTPUTS['authorization'])

stable_write_csv(
    OUTPUTS['input_inventory'],
    input_records,
    [
        'input_id',
        'source_cell',
        'path',
        'sha256',
        'bytes',
        'sidecar_path',
        'sidecar_valid',
        'row_level_content_opened_in_cell_7c3',
    ],
)
write_sidecar(OUTPUTS['input_inventory'])

print(f'Authorization decision: {authorization_decision}')
print('Cell 7C4 prompt materialization authorized : YES')
print('LLM execution authorized                   : NO')
print('Answer-key access authorized               : NO')

Authorization decision: AUTHORIZE_STAGE7C_CELL7C4_SCORE_BLIND_PROMPT_MATERIALIZATION_ONLY_480_QUESTION_ALIAS_PROMPTS_FIVE_FROZEN_CONTEXT_BLOCKS_EACH_EXACT_CELL7B4_TEMPLATES_NO_SCORE_BEARING_AUDIT_GES_QUALITY_RANK_RRF_CONDITION_IDENTITY_ANSWER_KEYS_LLM_OR_METRICS
Cell 7C4 prompt materialization authorized : YES
LLM execution authorized                   : NO
Answer-key access authorized               : NO


## 6. Final QC, immutable readback, and terminal decision

In [7]:
checks = OrderedDict([
    ('cell_7c2_all_7_hashes_exact', len(verified_7c2) == 7),
    ('cell_7c2_all_7_sidecars_valid', all(v['sidecar_valid'] for v in verified_7c2.values())),
    ('cell_7c2_terminal_pass_exact',
     manifest_7c2.get('terminal_decision') == EXPECTED_CELL_7C2_TERMINAL_DECISION),
    ('cell_7c2_next_cell_none', manifest_7c2.get('next_authorized_cell') is None),
    ('cell_7c2_qc_zero_failures', int(qc_7c2.get('failed_checks', -1)) == 0),
    ('score_blind_top5_rows_2400', top5_meta['rows'] == 2400),
    ('score_blind_top5_exact_schema', top5_meta['schema_names'] == expected_top5_columns),
    ('score_blind_top5_no_leakage_columns', len(leaking_top5_columns) == 0),
    ('score_bearing_audit_not_opened', True),
    ('semantic_corpus_exact_hash',
     sha256_file(resolved_7b3['semantic_corpus']) == CELL_7B3_SCORE_BLIND['semantic_corpus']['sha256']),
    ('semantic_corpus_rows_100920', semantic_meta['rows'] == 100_920),
    ('semantic_corpus_required_fields',
     required_semantic_columns.issubset(set(semantic_meta['schema_names']))),
    ('primary_questions_exact_hash',
     sha256_file(resolved_7b3['primary_questions']) == CELL_7B3_SCORE_BLIND['primary_questions']['sha256']),
    ('primary_questions_rows_80', question_rows == 80),
    ('primary_question_required_fields',
     {'question_id', 'question_text'}.issubset(set(question_header))),
    ('cell_7b4_all_6_hashes_exact', len(verified_7b4) == 6),
    ('cell_7b4_all_6_sidecars_valid', all(v['sidecar_valid'] for v in verified_7b4.values())),
    ('cell_7b4_terminal_pass_exact',
     manifest_7b4.get('terminal_decision') == EXPECTED_CELL_7B4_DECISION),
    ('all_frozen_prompt_config_checks_pass',
     all(bool(v) for v in frozen_prompt_checks.values())),
    ('authorization_targets_cell_7c4', AUTHORIZED_CELL['cell_id'] == '7C4'),
    ('authorization_once_true', AUTHORIZED_CELL['authorized_once'] is True),
    ('prompt_instances_480',
     authorization_payload['frozen_prompt_materialization_design']['prompt_instances'] == 480),
    ('context_blocks_per_prompt_5',
     authorization_payload['frozen_prompt_materialization_design']['context_blocks_per_prompt'] == 5),
    ('future_response_calls_1440',
     authorization_payload['frozen_prompt_materialization_design']['future_expected_response_calls_if_separately_authorized'] == 1440),
    ('score_values_in_prompt_false',
     authorization_payload['frozen_prompt_materialization_design']['score_values_in_prompt'] is False),
    ('condition_identity_in_prompt_false',
     authorization_payload['frozen_prompt_materialization_design']['condition_identity_in_prompt'] is False),
    ('answer_key_in_prompt_false',
     authorization_payload['frozen_prompt_materialization_design']['answer_key_in_prompt'] is False),
    ('llm_call_not_authorized',
     authorization_payload['frozen_prompt_materialization_design']['llm_call_authorized'] is False),
    ('cell_7a3_scores_not_loaded',
     authorization_payload['cell_7a3_scores_loaded'] is False),
    ('answer_keys_not_inspected',
     authorization_payload['answer_key_outcomes_inspected'] is False),
    ('prompts_not_materialized_in_7c3',
     authorization_payload['prompts_materialized'] is False),
    ('llm_not_called_in_7c3',
     authorization_payload['llm_called'] is False),
    ('rag_metrics_not_calculated',
     authorization_payload['rag_metrics_calculated'] is False),
])

failed = [name for name, passed in checks.items() if not bool(passed)]
if failed:
    raise RuntimeError(
        'Cell 7C3 authorization QC failed:\\n- '
        + '\\n- '.join(failed)
    )

terminal_decision = (
    'PASS_STAGE7C3_COMPLETE_CELL7C2_SCORE_BLIND_TOP5_CELL7B3_SEMANTIC_CORPUS_AND_QUESTIONS_'
    'AND_CELL7B4_PROMPT_LLM_CONFIGURATION_REVERIFIED_CHECKSUM_PROTECTED_CELL7C4_'
    '480_SCORE_BLIND_PROMPT_MATERIALIZATION_ONLY_AUTHORIZED_NO_SCORE_BEARING_AUDIT_GES_'
    'QUALITY_RANK_RRF_CONDITION_IDENTITY_ANSWER_KEYS_LLM_RESPONSES_ADJUDICATION_OR_RAG_METRICS'
)

qc_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'authorization_decision': authorization_decision,
    'passed_checks': len(checks),
    'failed_checks': 0,
    'total_checks': len(checks),
    'checks': [{'check': name, 'passed': bool(passed)} for name, passed in checks.items()],
    'scientific_operations': {
        'score_bearing_reranking_audit_loaded': False,
        'cell_7a3_scores_loaded': False,
        'prompts_materialized': False,
        'llm_called': False,
        'answer_keys_inspected': False,
        'adjudication_performed': False,
        'rag_metrics_calculated': False,
    },
    'terminal_decision': terminal_decision,
}
stable_write_json(OUTPUTS['qc'], qc_payload)
write_sidecar(OUTPUTS['qc'])

manifest_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'notebook': NOTEBOOK_NAME,
    'project_root': str(ROOT),
    'output_artifacts': {
        key: {
            'path': str(path),
            'sha256': sha256_file(path),
            'sidecar_path': str(sidecar_path(path)),
            'sidecar_valid': sidecar_is_valid(path),
        }
        for key, path in OUTPUTS.items()
        if key != 'manifest'
    },
    'authorization_decision': authorization_decision,
    'terminal_decision': terminal_decision,
    'next_authorized_cell': '7C4',
    'llm_execution_authorized': False,
    'next_required_action': (
        'Execute Cell 7C4 only to materialize and checksum-freeze 480 score-blind prompts. '
        'A separate authorization is required after Cell 7C4 before any LLM call.'
    ),
}
stable_write_json(OUTPUTS['manifest'], manifest_payload)
write_sidecar(OUTPUTS['manifest'])

# Fresh readback.
for key, path in OUTPUTS.items():
    if not path.exists():
        raise FileNotFoundError(f'Missing Cell 7C3 output: {path}')
    if not sidecar_is_valid(path):
        raise AssertionError(f'Cell 7C3 output sidecar failed: {path}')

auth_readback = load_json(OUTPUTS['authorization'])
qc_readback = load_json(OUTPUTS['qc'])
manifest_readback = load_json(OUTPUTS['manifest'])

readback_checks = OrderedDict([
    ('authorization_decision_readback_exact',
     auth_readback['authorization_decision'] == authorization_decision),
    ('authorized_cell_readback_7c4',
     auth_readback['authorized_cell']['cell_id'] == '7C4'),
    ('llm_authorization_false_readback',
     auth_readback['frozen_prompt_materialization_design']['llm_call_authorized'] is False),
    ('qc_zero_failures_readback',
     int(qc_readback['failed_checks']) == 0),
    ('manifest_terminal_decision_exact',
     manifest_readback['terminal_decision'] == terminal_decision),
    ('manifest_next_cell_7c4',
     manifest_readback['next_authorized_cell'] == '7C4'),
    ('manifest_llm_execution_false',
     manifest_readback['llm_execution_authorized'] is False),
    ('all_four_output_sidecars_valid',
     all(sidecar_is_valid(path) for path in OUTPUTS.values())),
])

failed_readback = [name for name, passed in readback_checks.items() if not bool(passed)]
if failed_readback:
    raise RuntimeError(
        'Cell 7C3 readback QC failed:\\n- '
        + '\\n- '.join(failed_readback)
    )

passed_checks = len(checks) + len(readback_checks)
total_checks = passed_checks

separator = '=' * 158
print('\\n' + separator)
print('EXPERIMENT 2 — STAGE 7C — CELL 7C3')
print('SCORE-BLIND PROMPT-MATERIALIZATION EXECUTION AUTHORIZATION FREEZE')
print(separator)
print(f'Notebook                                      : {NOTEBOOK_NAME}')
print(f'Project root                                  : {ROOT}')

print('\\nUPSTREAM CELL 7C2 REVERIFICATION')
print(f'Cell 7C2 manifest SHA-256                     : {sha256_file(CELL_7C2["manifest"]["path"])}')
print('Cell 7C2 terminal PASS verified               : YES')
print('Frozen Cell 7C2 artifacts                     : 7/7 exact hashes + sidecars')
print(f'Score-blind final top-5 rows                  : {top5_meta["rows"]:,}')
print('Score-bearing Cell 7C2 audit opened           : NO')

print('\\nSCORE-BLIND PROMPT INPUT REVERIFICATION')
print(f'Cell 7B3 semantic corpus SHA-256               : {sha256_file(resolved_7b3["semantic_corpus"])}')
print(f'Cell 7B3 primary questions SHA-256             : {sha256_file(resolved_7b3["primary_questions"])}')
print(f'Cell 7B4 prompt/response config SHA-256        : {sha256_file(CELL_7B4["llm_prompt_response"]["path"])}')
print(f'Cell 7B4 runtime config SHA-256                : {sha256_file(CELL_7B4["runtime_determinism"]["path"])}')
print(f'Cell 7B4 manifest SHA-256                      : {sha256_file(CELL_7B4["manifest"]["path"])}')
print('Frozen prompt templates/schema                : VERIFIED')
print('Frozen LLM snapshot                           : gpt-4.1-mini-2025-04-14')
print('Answer-key outcomes inspected                 : NO')

print('\\nCELL 7C4 AUTHORIZATION')
print('Prompt instances                              : 480 = 80 questions × 6 blinded aliases')
print('Context blocks per prompt                     : 5')
print('Context order                                 : frozen context_position 1 through 5')
print('Question position                             : before evidence context')
print('Score values in prompt                        : NO')
print('A-F condition identity in prompt              : NO')
print('Answer key in prompt                          : NO')
print('LLM call                                      : NOT AUTHORIZED')
print('Later repetitions if separately authorized    : 3 per prompt = 1,440 total calls')

print('\\nCELL 7C3 FROZEN OUTPUTS')
for label, path in OUTPUTS.items():
    print(f'{label:<46}: {path}')
    print(f'{"SHA-256":<46}: {sha256_file(path)}')

print(f'\\nQC checks                                      : {passed_checks}/{total_checks} PASS')

print('\\nSCIENTIFIC OPERATIONS IN CELL 7C3')
print('Prompts materialized                         : NO')
print('LLM called                                   : NO')
print('Answer-key outcomes inspected                : NO')
print('Adjudication or RAG metrics                  : NO')

print('\\nNEXT AUTHORIZED CELL')
print('Stage 7C — Cell 7C4                           : score-blind prompt materialization only')
print('LLM generation                                : PROHIBITED')
print('Score-bearing reranking audit                 : PROHIBITED')
print('Answer keys / metrics                         : PROHIBITED')

print(f'\\nFINAL DECISION                                : {terminal_decision}')
print(separator)

\n==============================================================================================================================================================
EXPERIMENT 2 — STAGE 7C — CELL 7C3
SCORE-BLIND PROMPT-MATERIALIZATION EXECUTION AUTHORIZATION FREEZE
Notebook                                      : 10_GES_Aware_Genomic_RAG_Cell_7C3_Prompt_Materialization_Execution_Authorization_V3.ipynb
Project root                                  : /content/drive/MyDrive/GES_RAG_Temporal_Study
\nUPSTREAM CELL 7C2 REVERIFICATION
Cell 7C2 manifest SHA-256                     : 8a8e3dc827b4641f28922b8712c3024fb4c82ba1e3b4511213f28b9b4b1a83d5
Cell 7C2 terminal PASS verified               : YES
Frozen Cell 7C2 artifacts                     : 7/7 exact hashes + sidecars
Score-blind final top-5 rows                  : 2,400
Score-bearing Cell 7C2 audit opened           : NO
\nSCORE-BLIND PROMPT INPUT REVERIFICATION
Cell 7B3 semantic corpus SHA-256               : 2fead04f6c0814bb87207c9c36db747537